# Stateless operators

Stateless operators are "stateless" in that they do not accumulate state.

Simply put, a topology only consisting of stateless operators can swallow arbitrary many messages from the sources without ever running out of memory.


## Overview

[Prepation](#prep)

* [map()](#map-operator)
* [peek()](#peek-operator)
* [flatmap()](#flatmap-operator)
* [filter()](#filter-operator)
* [merge()](#merge-operator)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [ ]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"


---
<a id="map-operator"></a>
## map()

Classic `map()` operator, like e.g. in Kafka Streams:

```python
def map(self, map_fun, **kwargs):
"""Transform each record into another record.

Args:
    map_fun: r -> r - map function
    **kwargs: passed through to the underlying node(s)
Returns:
    tn: the newly created topology node of the operator"""
```

Here is an example:

In [ ]:
tn = Tn.build(
    Tn.source(click_source_str)
    ###
    # map() operator: select value.customer_id as customer_id, and value.view_time as view_time
    ###
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

input_m_list = click_generator.generate(5)
print("Input:")
for m in input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


What happened? We pushed five example messages to the topology which uses `map()` to select `value.customer_id` as `customer_id`, and `value.view_time` as `view_time`.

This is how the topology looks like graphically:

```mermaid
graph TD
ba9b2d80-3f93-4b57-9eb6-d02ff933493d[source_clicks] --> 1ab0b693-3e7d-486c-a2d7-6596a8671cc9[map_op]
```

---
<a id="peek-operator"></a>
## peek()

This operator is mainly for debugging purposes. Under the covers, it's `map()` that returns the input record but allows you to trigger a side effect (such as a `print`) beforehand:

```python
def peek(self, prefix_str=None, peek_fun=None, **kwargs):
    """Cause a side effect on a record. The records pass through unchanged.
    
    Args:
        prefix_str: label printed before each record (if peek_fun is None; default: no prefix)
        peek_fun: r -> None - cause a side effect on record r (default if peek_fun is None: print)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here are a few examples. The first provides no arguments:

In [ ]:
tn = Tn.build(
    Tn.source(click_source_str)
    ###
    # peek() operator without arguments to just print out the records
    ###
    .peek()
)

m_list = click_generator.generate(5)

_ = tn.process({click_source_str: m_list})


You can see that `peek()` just printed out each of the records coming in.

Here is the topology graph - you can see that the `peek()` is actually nothing else but a `map()` under the covers:

```mermaid
graph TD
ba9b2d80-3f93-4b57-9eb6-d02ff933493d[source_clicks] --> 1ab0b693-3e7d-486c-a2d7-6596a8671cc9[map_op]
```

Next, we use `peek` with `prefix_str` set:

In [ ]:
tn = Tn.build(
    Tn.source(click_source_str)
    ###
    # peek() operator with prefix_str argument to print out the records with a prefix/label
    ###
    .peek("label")
)

m_list = click_generator.generate(5)

_ = tn.process({click_source_str: m_list})


...and you can see that `peek()` now adds the prefix `label: ` to the input records printed out.

Last example: We set the `peek_fun` parameter to be able to provide our own side effect:

In [ ]:
tn = Tn.build(
    Tn.source(click_source_str)
    ###
    # peek() operator with peek_fun argument to define our own side effect
    ###
    .peek(peek_fun=lambda r: print(f"{r}\n"))
)

m_list = click_generator.generate(5)

_ = tn.process({click_source_str: m_list})



What we did in the `peek_fun` is to also print out the input records and add a newline after each one of them.


---
<a id="flatmap-operator"></a>
## flatmap()

This is the relational version of the classic `flatmap` operator also known from e.g. Kafka Streams.

It is relational in the sense that its return value (an iterable) is interpreted as an unordered *set* of outputs instead of as an ordered *list*:

```python
def flatmap(self, flatmap_fun, **kwargs):
    """Explode each record into an iterable of records interpreted as an unordered set.
    
    Args:
        flatmap_fun: r -> iterable of r
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Examples to follow:

In [ ]:
tn = Tn.build(
    Tn.source(customer_source_str)
    ###
    # flapmap() operator - split the string in value.name by spaces and return the splits as a set
    ###
    .flatmap(lambda r: {name_part_str for name_part_str in r["value"]["name"].split(" ")})
)

input_m_list = customer_generator.generate(5)
print("Input:")
for m in input_m_list:
    print(m)

output_m_list = tn.process({customer_source_str: input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In this example, from each customer input record, we take the `name` field of its `value`, split it and return the set of the parts of the name. So e.g., `"Alexis Jones"` becomes `{"Alexis", "Jones"}`.

Here is the Mermaid representation of this topology:
```mermaid
graph TD
f2306785-a69c-4dfc-8ad3-4e8fd593dd4a[source_customers] --> 9ad0babc-b672-4d6b-a968-f33ab51cb164[flatmap_op]
```

A classical flatmap returns a list. In Kafi Streams, being based on a relational engine that is pydbsp, it is an iterable (list or set etc.).

The key point is that this iterable is always interpreted as a *set*:
* the order of the resulting iterable is not preserved
* duplicates are automatically removed

To see this clearly, look at the following example where we stitch together an input message where the first name is the same as the last name, and we add a third name component as well:

In [ ]:
tn = Tn.build(
    Tn.source(customer_source_str)
    .flatmap(lambda r: {name_part_str for name_part_str in r["value"]["name"].split(" ")})
)

input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Frank Frank Anna'}}]
print("Input:")
print(input_m_list)

output_m_list = tn.process({customer_source_str: input_m_list})
print("\nOutput:")
print(output_m_list)


As you can observe:
* the order of the input string (`Frank` came in before `Anna`) is not preserved,
* the duplicate occurrence of `Frank` is automatically de-duplicated

Welcome to set theory ;-)

If you miss your "classical" list-returning flatmap, don't despair - you can still recover it e.g. as below:

In [ ]:
tn = Tn.build(
    Tn.source(customer_source_str)
    ###
    # flatmap() operator - add position of the splits
    ###
    .flatmap(lambda r: {(i, name_part_str) for i, name_part_str in enumerate(r["value"]["name"].split(" "))})
)

input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Frank Frank Anna'}}]
print("Input:")
print(input_m_list)

output_m_list = tn.process({customer_source_str: input_m_list})
print("\nOutput:")
print(output_m_list)


---
<a id="filter-operator"></a>
## filter()

On to the next classical stateless operator: `filter`:

```python
def filter(self, filter_fun, **kwargs):
    """Filter records according to a predicate.
    
    Args:
        filter_fun: r -> bool - filter predicate
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:


In [ ]:
tn = Tn.build(
    Tn.source(click_source_str)
    ###
    # filter() operator - keep only those records where value.view_time > 60
    ###
    .filter(lambda r: r["value"]["view_time"] > 60)
)

input_m_list = click_generator.generate(5)
print("Input:")
for m in input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In the example, we just keep those records whose `view_time` is greater than `60`.

Here is the Mermaid representation of the topology:
```mermaid
graph TD
0dab1488-2505-4a9d-a9f6-7294dc3e42c8[source_clicks] --> 54ad4109-956d-404b-9520-04fd10d1bf20[filter_op]
```

---
<a id="merge-operator"></a>
## merge()

Similar to Kafka Streams, the purpose of `merge` is to combine two branches of your topology into one:

```python
def merge(self, other_tn, **kwargs):
    """Merge two topology nodes.
    
    Args:
        other_tn: the other topology node
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

This screams for an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"id": r["value"]["customer_id"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"]})
)

tn = Tn.build(
    click_tn
    ###
    # merge() operator - merge click_tn and customer_tn
    ###
    .merge(customer_tn)
)

click_input_m_list = click_generator.generate(5)
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = customer_generator.generate(5)
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


This is what happens:
* We create two sub topologies - one for the clicks (`click_tn`) and one for the customers (`customer_tn`).
* In each sub topology, we select just the customer ID from the input records (`customer_id` for the clicks, `id` for the customers).
* We then use the `merge()` operator to merge the outputs of the two sub topologies together.

Here is the Mermaid representation:

```mermaid
graph TD
da7fe5fc-a151-4e68-8eaa-968befa3493d[source_clicks] --> 95269802-5d4f-4da4-ad3b-7d8d1b01180f[map_op]
3b8e544c-2496-41e0-965c-e08ecb3e0851[source_customers] --> 3c9d2262-43b3-4b45-b669-38930e52f349[map_op]
95269802-5d4f-4da4-ad3b-7d8d1b01180f[map_op] --> c9967e4e-f8e6-40e7-835d-a588f3df2cf1[merge_op]
3c9d2262-43b3-4b45-b669-38930e52f349[map_op] --> c9967e4e-f8e6-40e7-835d-a588f3df2cf1[merge_op]
```

`merge` is a stateless operation. It is neither a union or a join.

Instead, under the covers, it just adds up the weights of the ZSets of the input records:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"id": r["value"]["customer_id"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"]})
)

tn = Tn.build(
    click_tn
    .merge(customer_tn)
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 76, 'ts': 1786618958910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In the latest example, we just sent one message to each source, both having the same customer ID (`42`). The result are *two* records, not one (under the covers, it's actually one record with weight `2`).